# Marchenko-Pastur fit til transformer vektmatriser

**Kjør på Colab T4 (gratis) — GPT-2 er standard, tar ~2 min**

## Hva vi tester

Marchenko-Pastur (MP) er nullhypotesen for tilfeldige matriser:
singulærverdiene til en tilfeldig m×n matrise følger MP-fordelingen.

For en trent transformermodell forventer vi:
- **Spikes** over MP-kanten: singulærverdier som bærer signal (struktur)
- **Bulk**: singulærverdier som matcher MP (støy)

**Framleis-prediksjon**: tau og spike-fraksjon korrelerer.
Lag med høy tau (strukturell koherens) skal ha flere spikes over MP-kanten.

**Goldilocks**: [exp(−γ), 1/ζ(3)] = [0.5615, 0.8319]

---
Referanse: Martin & Mahoney (2019) — Implicit Self-Regularization in Deep Neural Networks


In [ ]:
!pip install -q transformers torch numpy scipy matplotlib

In [ ]:
import torch
import numpy as np
import math
import scipy.stats
import matplotlib.pyplot as plt
from transformers import AutoModel

GAMMA_EC = 0.5772156649
ZETA3    = 1.2020569032
TAU_MIN  = math.exp(-GAMMA_EC)   # 0.5615
TAU_MAX  = 1.0 / ZETA3           # 0.8319


def mp_edges(s, m, n):
    """
    Marchenko-Pastur øvre og nedre kant for singulærverdier s av en m×n matrise.

    Derivasjon:
      q     = min(m,n)/max(m,n)          # aspektforhold <= 1
      sigma2 = sum(s**2) / min(m,n)      # skalert variansestimat
      s+/-  = sqrt(sigma2) * (1 +/- sqrt(q))
    """
    q      = min(m, n) / max(m, n)
    sigma2 = np.sum(s**2) / min(m, n)
    s_plus  = math.sqrt(sigma2) * (1 + math.sqrt(q))
    s_minus = math.sqrt(sigma2) * (1 - math.sqrt(q))
    return s_plus, s_minus, q


def compute_tau(s):
    """tau = exp(H_spektral) / rank_max  (H = Shannon-entropi av normalisert s2)"""
    s2  = s**2
    p   = s2 / s2.sum()
    p   = p[p > 1e-12]
    H   = -np.sum(p * np.log(p))
    return math.exp(H) / len(s)


def layer_stats(W_np, name=""):
    """Beregn tau, MP-kanter og spike-fraksjon for en 2D vektmatrise."""
    m, n = W_np.shape
    s = np.linalg.svd(W_np, compute_uv=False)

    tau                  = compute_tau(s)
    s_plus, s_minus, q   = mp_edges(s, m, n)
    n_spikes             = int(np.sum(s > s_plus))
    spike_frac           = n_spikes / len(s)
    bulk_frac            = 1.0 - spike_frac

    return {
        'name':       name,
        'shape':      (m, n),
        'rank':       len(s),
        'tau':        tau,
        's_plus':     s_plus,
        's_minus':    s_minus,
        'q':          q,
        'n_spikes':   n_spikes,
        'spike_frac': spike_frac,
        'bulk_frac':  bulk_frac,
        's':          s,
    }


print(f"Goldilocks-sone: [{TAU_MIN:.4f}, {TAU_MAX:.4f}]")
print('Funksjoner klare.')

In [ ]:
# Velg modell — bytt til 'gpt2-medium', 'gpt2-large' eller 'Qwen/Qwen2.5-7B' for større test
# GPT-2 (117M) er default: ingen autentisering, ferdig på ~90 sek på T4
MODEL_NAME = "gpt2"

print(f"Laster {MODEL_NAME}...")
model = AutoModel.from_pretrained(MODEL_NAME, output_hidden_states=True)
model.eval()
print('Klar.')

# Tell 2D vektmatriser
n_matrices = sum(1 for _, p in model.named_parameters()
                 if p.dim() == 2 and min(p.shape) >= 8)
print(f"Antall 2D-vektmatriser (>=8 i begge dim): {n_matrices}")

In [ ]:
# Ekstraher og analyser alle 2D vektmatriser
# SVD er O(mn2) — GPT-2 tar ~1-2 min på T4

results = []

for name, param in model.named_parameters():
    if param.dim() != 2:
        continue
    m, n = param.shape
    if min(m, n) < 8:       # hopp over embedding-lignende smale matriser
        continue

    W = param.detach().float().numpy()
    stats = layer_stats(W, name=name)
    results.append(stats)

    in_gold = TAU_MIN <= stats['tau'] <= TAU_MAX
    tag = 'GOLD *' if in_gold else ''
    print(f"{name:<55} tau={stats['tau']:.4f}  spikes={stats['n_spikes']:3d}/{stats['rank']:3d}  {tag}")

print(f"\nTotalt analysert: {len(results)} matriser")

In [ ]:
# Oppsummering per lagtype
from collections import defaultdict

def lag_type(name):
    if 'attn' in name and any(x in name for x in ['c_attn','q_proj','k_proj','v_proj']):
        return 'attn_qkv'
    if 'attn' in name and any(x in name for x in ['c_proj','out_proj']):
        return 'attn_out'
    if 'mlp' in name and any(x in name for x in ['c_fc','fc_in','up','gate']):
        return 'mlp_up'
    if 'mlp' in name and any(x in name for x in ['c_proj','fc_out','down']):
        return 'mlp_down'
    return 'other'

by_type = defaultdict(list)
for r in results:
    by_type[lag_type(r['name'])].append(r)

print('Gjennomsnitt per lagtype:')
print(f"{'Type':<15}  {'N':>4}  {'tau':>8}  {'spike_frac':>11}  {'n_spikes':>9}")
print('-' * 55)
for t, rs in sorted(by_type.items()):
    tau_m   = np.mean([r['tau'] for r in rs])
    spk_m   = np.mean([r['spike_frac'] for r in rs])
    nspk_m  = np.mean([r['n_spikes'] for r in rs])
    print(f"{t:<15}  {len(rs):>4}  {tau_m:>8.4f}  {spk_m:>11.4f}  {nspk_m:>9.1f}")

all_tau   = [r['tau'] for r in results]
all_spike = [r['spike_frac'] for r in results]
print()
print(f"Samlet tau:         min={min(all_tau):.4f}  median={np.median(all_tau):.4f}  max={max(all_tau):.4f}")
print(f"Samlet spike_frac:  min={min(all_spike):.4f}  median={np.median(all_spike):.4f}  max={max(all_spike):.4f}")
n_in_gold = sum(1 for t in all_tau if TAU_MIN <= t <= TAU_MAX)
print(f"\nMatriser i Goldilocks-sona: {n_in_gold}/{len(results)} ({100*n_in_gold/len(results):.1f}%)")


In [ ]:
# Korrelasjonstest: tau vs spike_frac
# Framleis-prediksjon: høy tau <-> mange spikes (strukturerte lag)

from scipy.stats import pearsonr, spearmanr

tau_arr   = np.array(all_tau)
spike_arr = np.array(all_spike)

r_p, p_p = pearsonr(tau_arr, spike_arr)
r_s, p_s = spearmanr(tau_arr, spike_arr)

print('Korrelasjon: tau vs spike_frac')
print(f'  Pearson r  = {r_p:+.4f}  (p={p_p:.4f})')
print(f'  Spearman r = {r_s:+.4f}  (p={p_s:.4f})')
print()
if abs(r_s) > 0.3 and p_s < 0.05:
    retning = 'POSITIV' if r_s > 0 else 'NEGATIV'
    print(f'Signifikant {retning} korrelasjon: Framleis-prediksjon {"STØTTET ✓" if r_s > 0 else "MOTBEVIST ✗"}')
    print('  (høy tau <-> mange spikes = lag bærer strukturell informasjon utover MP-støy)')
else:
    print('Ingen signifikant korrelasjon.')
    print('  Enten er modellen tilfeldig (usannsynlig for trent modell),')
    print('  eller tau og spike_frac måler ulike aspekter av struktur.')

In [ ]:
# Plot 1: tau vs spike_frac scatter + tau-histogram
colors = {'attn_qkv': '#2196F3', 'attn_out': '#03A9F4',
          'mlp_up': '#FF5722', 'mlp_down': '#FF9800', 'other': '#9E9E9E'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for t, rs in by_type.items():
    ax.scatter([r['tau'] for r in rs],
               [r['spike_frac'] for r in rs],
               label=t, color=colors.get(t, '#666666'), alpha=0.7, s=40)
ax.axvline(TAU_MIN, color='green', linestyle='--', linewidth=0.8, alpha=0.6, label='Goldilocks')
ax.axvline(TAU_MAX, color='green', linestyle='--', linewidth=0.8, alpha=0.6)
ax.set_xlabel('tau (spektral koherens)')
ax.set_ylabel('spike_frac (fraksjon over MP-kant)')
ax.set_title(f'{MODEL_NAME}: tau vs spike_frac\nSpearman r={r_s:.3f} p={p_s:.3f}')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.hist(tau_arr, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
ax2.axvline(TAU_MIN, color='green', linestyle='--', linewidth=1.2, label=f'tau_min={TAU_MIN:.4f}')
ax2.axvline(TAU_MAX, color='darkgreen', linestyle='--', linewidth=1.2, label=f'tau_max={TAU_MAX:.4f}')
ax2.axvline(np.median(tau_arr), color='red', linestyle='-', linewidth=1.2, label=f'median={np.median(tau_arr):.4f}')
ax2.set_xlabel('tau')
ax2.set_ylabel('antall matriser')
ax2.set_title('Tau-fordeling: vektmatriser')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('tau_mp_scatter.png', dpi=120, bbox_inches='tight')
plt.show()
print('Lagret: tau_mp_scatter.png')

In [ ]:
# Plot 2: Singulærverdi-spekter for laget med flest spikes
best = max(results, key=lambda r: r['n_spikes'])
s    = best['s']
m, n = best['shape']

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(s, bins=60, density=True, color='steelblue', alpha=0.6, label='Empirisk')
ax.axvline(best['s_plus'],  color='red',    linestyle='--', linewidth=1.5,
           label=f"MP s_max = {best['s_plus']:.3f}")
ax.axvline(best['s_minus'], color='orange', linestyle='--', linewidth=1.2,
           label=f"MP s_min = {best['s_minus']:.3f}")
spikes = s[s > best['s_plus']]
ax.scatter(spikes, np.zeros_like(spikes) + 0.01,
           color='red', zorder=5, s=20, label=f'{len(spikes)} spikes')
ax.set_xlabel('Singulærverdi')
ax.set_ylabel('Tetthet')
ax.set_title(f"MP-fit: {best['name']}\n"
             f"tau={best['tau']:.4f}  spike_frac={best['spike_frac']:.4f}  q={best['q']:.3f}")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('tau_mp_spectrum.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Lag med flest spikes: {best['name']}")
print(f"  Shape: {best['shape']}  q={best['q']:.3f}")
print(f"  tau={best['tau']:.4f}  spikes={best['n_spikes']}/{best['rank']}")

In [ ]:
# E[tau] under MP-nullhypotesen — Monte Carlo
# Spørsmål til Percy Deift: hva er E[tau] under MP(q, sigma2=1)?

def tau_mp_null(q, n_samples=500, n_rep=200):
    n = n_samples
    m = max(1, int(round(q * n)))
    taus = []
    for _ in range(n_rep):
        W  = np.random.randn(m, n) / math.sqrt(n)
        s  = np.linalg.svd(W, compute_uv=False)
        taus.append(compute_tau(s))
    return np.mean(taus), np.std(taus)

print('E[tau] under MP-nullhypotesen (Monte Carlo, n_rep=200):')
print(f"{'q':>6}  {'E[tau]':>8}  {'std':>7}")
print('-' * 28)
for q_test in [0.1, 0.25, 0.5, 0.75, 1.0]:
    mu, sd = tau_mp_null(q_test)
    in_g   = TAU_MIN <= mu <= TAU_MAX
    print(f"{q_test:>6.2f}  {mu:>8.4f}  {sd:>7.4f}  {'GOLD' if in_g else ''}")
print()
print('Faktisk median tau for denne modellen:', f"{np.median(tau_arr):.4f}")
print('Framleis-prediksjon: trent modell avviker fra MP-null (høyere tau = mer struktur)')

In [ ]:
# Samlet konklusjon
print('=' * 60)
print(f'MODELL: {MODEL_NAME}')
print('=' * 60)
print(f'Matriser analysert:      {len(results)}')
print(f'tau median:              {np.median(tau_arr):.4f}')
print(f'tau i Goldilocks:        {n_in_gold}/{len(results)} ({100*n_in_gold/len(results):.1f}%)')
print(f'spike_frac median:       {np.median(spike_arr):.4f}')
print(f'Pearson r (tau,spike):   {r_p:+.4f}  p={p_p:.4f}')
print(f'Spearman r (tau,spike):  {r_s:+.4f}  p={p_s:.4f}')
print()
print('Lagtype med høyest tau:')
best_type = max(by_type.items(), key=lambda kv: np.mean([r['tau'] for r in kv[1]]))
print(f'  {best_type[0]}: tau={np.mean([r["tau"] for r in best_type[1]]):.4f}')
print()
print('Lagtype med flest spikes:')
most_spikes = max(by_type.items(), key=lambda kv: np.mean([r['spike_frac'] for r in kv[1]]))
print(f'  {most_spikes[0]}: spike_frac={np.mean([r["spike_frac"] for r in most_spikes[1]]):.4f}')
print('=' * 60)